# Dans ce premier notebook, nous allons télécharger et nettoyer les données

In [1]:
#!pip install lxml
#!pip install html5lib
#!pip install bs4

In [2]:
import os
import tarfile
import urllib.request
import pandas as pd

# URLs et chemins
TAR_URL = "https://www.phase-trans.msm.cam.ac.uk/map/data/tar/welddb.tar"
DATA_DIR = os.path.join("..", "data")
TAR_PATH = os.path.join(DATA_DIR, "welddb.tar")
DATA_PATH = os.path.join(DATA_DIR, "welddb.data")

# 1. Création du dossier et téléchargement
os.makedirs(DATA_DIR, exist_ok=True)
req = urllib.request.Request(TAR_URL, headers={"User-Agent": "Mozilla/5.0"})

with urllib.request.urlopen(req) as response, open(TAR_PATH, "wb") as f:
    f.write(response.read())

# 2. Extraction du fichier welddb.data
with tarfile.open(TAR_PATH) as tar:
    tar.extract("welddb.data", path=DATA_DIR)

# 3. Chargement dans Pandas (gestion des valeurs manquantes 'N' et '-')
df = pd.read_csv(DATA_PATH, sep=r"\s+", na_values=["N", "-"])

print(f"Dataset chargé avec succès ! Dimensions : {df.shape}")
df.head()

Dataset chargé avec succès ! Dimensions : (1651, 44)


,0.037,0.30,0.65,0.008,0.012,0,N,N.1,N.2,N.3,...,N.15,N.16,N.17,N.18,N.19,N.20,N.21,N.22,N.23,Evans-Ni/CMn-1990/1991-0Aaw
0,0.037,0.30,0.65,0.008,0.012,0.0,NaN,NaN,NaN,NaN,...,-28.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Evans-Ni/CMn-1990/1991-0Aawch
1,0.037,0.30,0.65,0.008,0.012,0.0,NaN,NaN,NaN,NaN,...,-38.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Evans-Ni/CMn-1990/1991-0Aht
2,0.037,0.31,1.03,0.007,0.014,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Evans-Ni/CMn-1990/1991-0Baw
3,0.037,0.31,1.03,0.007,0.014,0.0,NaN,NaN,NaN,NaN,...,-48.0,100.0,NaN,NaN,32,28.0,40.0,0.0,0.0,Evans-Ni/CMn-1990/1991-0Bawch
4,0.037,0.31,1.03,0.007,0.014,0.0,NaN,NaN,NaN,NaN,...,-44.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Evans-Ni/CMn-1990/1991-0Bht


On voit que les différentes features n'ont pas des noms clairs, on va les renommer avec le fichier .info qui accompagne la db

In [3]:
# Dictionnaire ou liste des 44 noms de colonnes explicites
columns_names = [
    ### Composition chimique
    #wt% pourcentages massiques, si nulle, on peut dire 0
    "C_wt",          # Col 1: Carbon (wt%)
    "Si_wt",         # Col 2: Silicon (wt%)
    "Mn_wt",         # Col 3: Manganese (wt%)
    "S_wt",          # Col 4: Sulphur (wt%)
    "P_wt",          # Col 5: Phosphorus (wt%)
    "Ni_wt",         # Col 6: Nickel (wt%)
    "Cr_wt",         # Col 7: Chromium (wt%)
    "Mo_wt",         # Col 8: Molybdenum (wt%)
    "V_wt",          # Col 9: Vanadium (wt%)
    "Cu_wt",         # Col 10: Copper (wt%)
    "Co_wt",         # Col 11: Cobalt (wt%)
    "W_wt",          # Col 12: Tungsten (wt%)
    
    #Elements mineurs (traces)
    "O_ppm",         # Col 13: Oxygen (ppm)
    "Ti_ppm",        # Col 14: Titanium (ppm)
    "N_ppm",         # Col 15: Nitrogen (ppm)
    "Al_ppm",        # Col 16: Aluminium (ppm)
    "B_ppm",         # Col 17: Boron (ppm)
    "Nb_ppm",        # Col 18: Niobium (ppm)
    "Sn_ppm",        # Col 19: Tin (ppm)
    "As_ppm",        # Col 20: Arsenic (ppm)
    "Sb_ppm",        # Col 21: Antimony (ppm)
    
    #Conditions physiques liées à la soudure
    "Current_A",     # Col 22: Current (A)
    "Voltage_V",     # Col 23: Voltage (V)
    "AC_DC",         # Col 24: AC or DC
    "Polarity",      # Col 25: Electrode positive or negative
    "Heat_input",    # Col 26: Heat input (kJ/mm)
    "Interpass_temp",# Col 27: Interpass temperature (°C)
    "Weld_type",     # Col 28: Type of weld
    "PWHT_temp",     # Col 29: Post weld heat treatment temp (°C)
    "PWHT_time",     # Col 30: Post weld heat treatment time (h)
    
    #Test de la soudure 
    "Yield_strength",# Col 31: Yield strength (MPa)
    "UTS",           # Col 32: Ultimate tensile strength (MPa)
    "Elongation",    # Col 33: Elongation (%)
    "Reduction_area",# Col 34: Reduction of Area (%)
    "Charpy_temp",   # Col 35: Charpy temperature (°C)
    "Charpy_energy", # Col 36: Charpy impact toughness (J)
    "Hardness",      # Col 37: Hardness (kg/mm2)
    
    "FATT_50",       # Col 38: 50% FATT
    "Primary_ferrite",# Col 39: Primary ferrite in microstructure (%)
    "Ferrite_2nd_phase",# Col 40: Ferrite with second phase (%)
    "Acicular_ferrite",# Col 41: Acicular ferrite (%)
    "Martensite",    # Col 42: Martensite (%)
    "Ferrite_carbide",# Col 43: Ferrite with carbide aggregate (%)
    
    #Identifiant, pas pertinent pour le ML
    "Weld_ID"        # Col 44: Weld ID
]

# Attribution des nouveaux noms de colonnes
df.columns = columns_names

# Verification
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1651 entries, 0 to 1650
Data columns (total 44 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   C_wt               1651 non-null   float64
 1   Si_wt              1651 non-null   float64
 2   Mn_wt              1651 non-null   float64
 3   S_wt               1647 non-null   str    
 4   P_wt               1641 non-null   float64
 5   Ni_wt              696 non-null    float64
 6   Cr_wt              784 non-null    float64
 7   Mo_wt              793 non-null    str    
 8   V_wt               928 non-null    str    
 9   Cu_wt              578 non-null    str    
 10  Co_wt              129 non-null    str    
 11  W_wt               75 non-null     str    
 12  O_ppm              1256 non-null   float64
 13  Ti_ppm             935 non-null    str    
 14  N_ppm              1242 non-null   str    
 15  Al_ppm             905 non-null    str    
 16  B_ppm              504 non-null    

On voit qu'il y a pas mal de valeurs manquantes ainsi que des variables nominales, on va maintenant gérer ce problème. Différents traitement sont appliqués car une variable manquant ne signifie pas forcément la même chose en fonction du type de variable (par ex : si un élement mineur à une valeur nulle, on peut considérer qu'il n'a pas été détecté on peut donc remplacer par 0, alors que si le Yield_strengh manque, c'est que les tests n'ont pas été fait (on ne peut pas remplacer par 0))